In [1]:
# 매번 실행되도록 상단에 위치
!pip install numpy==1.23.5 --force-reinstall
!pip install pandas_ta
!pip install optuna


  Using cached numpy-1.23.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.3 kB)
Using cached numpy-1.23.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.1 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.23.5
    Uninstalling numpy-1.23.5:
      Successfully uninstalled numpy-1.23.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 1.23.5 which is incompatible.
albumentations 2.0.7 requires numpy>=1.24.4, but you have numpy 1.23.5 which is incompatible.
imbalanced-learn 0.13.0 requires numpy<3,>=1.24.3, but you have numpy 1.23.5 which is incompatible.
db-dtypes 1.4.3 requires numpy>=1.24.0, but you have numpy 1.23.5 which is incompatible.
albucore 0.0.24 requires numpy>=1.24.4, but you have numpy 1.23.5 which is incompatible.
tree

  Using cached pandas_ta-0.3.14b.tar.gz (115 kB)
  Preparing metadata (setup.py) ... done
  Created wheel for pandas_ta: filename=pandas_ta-0.3.14b0-py3-none-any.whl size=218910 sha256=7c0885f3baa3452d97f6afd827b465ab9aab50f8443a352d4bb5905760208ff7
  Stored in directory: /root/.cache/pip/wheels/7f/33/8b/50b245c5c65433cd8f5cb24ac15d97e5a3db2d41a8b6ae957d
Successfully built pandas_ta
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 26.3 MB/s eta 0:00:00


In [1]:
import pandas as pd
import json

# 웹에서 S&P 500 데이터 가져오기
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
tables = pd.read_html(url)
df = tables[0]

# 티커 및 이름 정리
ticker_Group = [t.replace(".", "-") for t in df['Symbol']]
company_names = [n.replace(".", "-") for n in df['Security']]
ticker_names = dict(zip(ticker_Group, company_names))

# 티커 리스트 저장
with open("ticker_Group.txt", "w") as f:
    for t in ticker_Group:
        f.write(f"{t}\n")

# 티커-이름 딕셔너리 저장 (json)
with open("ticker_names.json", "w") as f:
    json.dump(ticker_names, f, indent=2, ensure_ascii=False)

In [2]:
import pandas as pd

# 위키피디아의 S&P 500 구성 종목 목록 URL
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# HTML 테이블을 읽어오기
tables = pd.read_html(url)
df = tables[0]  # 첫 번째 테이블이 S&P 500 구성 종목 목록


# 티커와 기업명 추출
ticker_Group= df['Symbol'].tolist()
company_names = df['Security'].tolist()

ticker_Group = [t.replace(".", "-") for t in ticker_Group]
company_names = [t.replace(".", "-") for t in company_names]
ticker_names = dict(zip(ticker_Group, company_names))

ticker_Group
ticker_names

{'MMM': '3M',
 'AOS': 'A- O- Smith',
 'ABT': 'Abbott Laboratories',
 'ABBV': 'AbbVie',
 'ACN': 'Accenture',
 'ADBE': 'Adobe Inc-',
 'AMD': 'Advanced Micro Devices',
 'AES': 'AES Corporation',
 'AFL': 'Aflac',
 'A': 'Agilent Technologies',
 'APD': 'Air Products',
 'ABNB': 'Airbnb',
 'AKAM': 'Akamai Technologies',
 'ALB': 'Albemarle Corporation',
 'ARE': 'Alexandria Real Estate Equities',
 'ALGN': 'Align Technology',
 'ALLE': 'Allegion',
 'LNT': 'Alliant Energy',
 'ALL': 'Allstate',
 'GOOGL': 'Alphabet Inc- (Class A)',
 'GOOG': 'Alphabet Inc- (Class C)',
 'MO': 'Altria',
 'AMZN': 'Amazon',
 'AMCR': 'Amcor',
 'AEE': 'Ameren',
 'AEP': 'American Electric Power',
 'AXP': 'American Express',
 'AIG': 'American International Group',
 'AMT': 'American Tower',
 'AWK': 'American Water Works',
 'AMP': 'Ameriprise Financial',
 'AME': 'Ametek',
 'AMGN': 'Amgen',
 'APH': 'Amphenol',
 'ADI': 'Analog Devices',
 'ANSS': 'Ansys',
 'AON': 'Aon plc',
 'APA': 'APA Corporation',
 'APO': 'Apollo Global Managem

주식 정보 모으기


In [3]:
import yfinance as yf
import pandas as pd
import pandas_ta as ta
import numpy as np
NaN = np.nan
def get_value_safe(df, column_names):
    for col in column_names:
        if col in df.columns:
            series = df[col].dropna()
            if not series.empty:
                return series.iloc[0]
    return np.nan

def get_valuation_ratios_for_date(ticker_obj, info, fin, bal, shares, date):
    fin_q = fin.loc[fin.index <= date]
    bal_q = bal.loc[bal.index <= date]

    try:
        if fin_q.empty or bal_q.empty:
            raise ValueError("📉 재무제표 없음")

        # ✅ 가장 최근 분기 1개만 사용
        recent_fin = fin_q.iloc[-1]
        recent_bal = bal_q.iloc[-1]

        net_income = recent_fin.get("Net Income", np.nan)
        revenue = recent_fin.get("Total Revenue", np.nan)

        equity = get_value_safe(pd.DataFrame([recent_bal]), [
            'Total Equity Gross Minority Interest', 'Total Stockholder Equity'])
        liabilities = get_value_safe(pd.DataFrame([recent_bal]), [
            'Total Liabilities Net Minority Interest', 'Total Liabilities'])
        current_assets = get_value_safe(pd.DataFrame([recent_bal]), ['Total Current Assets'])
        current_liabilities = get_value_safe(pd.DataFrame([recent_bal]), ['Total Current Liabilities'])

        # 시가총액 계산
        market_cap = info['previousClose'] * shares if shares and info.get('previousClose') else np.nan

        # 지표 계산
        per = market_cap / net_income if pd.notna(market_cap) and pd.notna(net_income) and net_income != 0 else np.nan
        pbr = market_cap / equity if pd.notna(market_cap) and pd.notna(equity) and equity != 0 else np.nan
        psr = market_cap / revenue if pd.notna(market_cap) and pd.notna(revenue) and revenue != 0 else np.nan
        debt_equity = liabilities / equity if pd.notna(liabilities) and pd.notna(equity) and equity != 0 else np.nan

        # 유동비율
        if pd.notna(current_assets) and pd.notna(current_liabilities):
            current_ratio = current_assets / current_liabilities if current_liabilities != 0 else np.inf
        else:
            current_ratio = 0.01

        return {
            'PER': per,
            'PBR': pbr,
            'PSR': psr,
            'Current_Ratio': current_ratio,
            'Debt_Equity': debt_equity
        }

    except Exception as e:
        print("💥 계산 중 오류:", e)
        return {
            'PER': np.nan,
            'PBR': np.nan,
            'PSR': np.nan,
            'Current_Ratio': np.nan,
            'Debt_Equity': np.nan
        }

def check_buy_signal(ticker_symbol):
    print(f"{ticker_symbol}...")
    ticker = yf.Ticker(ticker_symbol)
    df = ticker.history(period="30mo")
    df['Ticker'] = ticker_symbol

    # 기술적 지표: 모두 shift(1)로 미래 누수 방지
    df["RSI"] = ta.rsi(df["Close"], length=5).shift(1)

    bb = ta.bbands(df["Close"], length=10, std=2.0)
    for col in bb.columns:
        df[col] = bb[col].shift(1)
    df['SMA5'] = df['Close'].rolling(window=5).mean().shift(1)
    df['SMA10'] = df['Close'].rolling(window=10).mean().shift(1)
    df['BB_Dist'] = df['Close'] - df['BBL_10_2.0']

    macd = ta.macd(df["Close"], fast=6, slow=13, signal=4)
    for col in macd.columns:
        df[col] = macd[col].shift(1)

    stoch = ta.stoch(df["High"], df["Low"], df["Close"], k=5, d=3, smooth_k=1)
    for col in stoch.columns:
        df[col] = stoch[col].shift(1)

    df["OBV"] = ta.obv(df["Close"], df["Volume"]).shift(1)
    # ATR (Average True Range) 추가: 변동성 지표
    df["ATR"] = ta.atr(df["High"], df["Low"], df["Close"], length=14).shift(1)
    df['Golden_Cross'] = (
        (df['SMA5'].shift(1) < df['SMA10'].shift(1)) &
        (df['SMA5'] >= df['SMA10'])
    ).astype(int)
    df["ADX"] = ta.adx(df["High"], df["Low"], df["Close"]).iloc[:, 0].shift(1)
    df["CCI"] = ta.cci(df["High"], df["Low"], df["Close"], length=20).shift(1)
    df["ROC"] = ta.roc(df["Close"], length=10).shift(1)
    df["CMF"] = ta.cmf(df["High"], df["Low"], df["Close"], df["Volume"], length=20).shift(1)
    df["ADL"] = ta.ad(df["High"], df["Low"], df["Close"], df["Volume"]).shift(1)



    df["Yesterday_Close"] = df["Close"].shift(1)
    df["Target"] = (df["Yesterday_Close"] <= df["Close"]).astype(int)

    df.reset_index(inplace=True)
    df['Date'] = df['Date'].dt.date

    # Drop NaNs
    df = df.dropna(subset=[
        "RSI", "BBL_10_2.0", "BBU_10_2.0", "SMA5", "SMA10",
        "MACD_6_13_4", "MACDh_6_13_4", "STOCHk_5_3_1", "STOCHd_5_3_1", "OBV","ATR"
    ])
    # 불필요한 열 제거
    df = df.drop(columns=[
      'BBL_10_2.0','BBM_10_2.0','BBU_10_2.0',
      'BBB_10_2.0','BBP_10_2.0', "SMA5", "SMA10"
    ])

    return df


all_dfs = []
for t in ticker_Group:
    df = check_buy_signal(t)
    if df is not None:
        all_dfs.append(df)

combined_df = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()

from datetime import datetime, time
import pytz

def is_kst_predict_time():
    now_kst = datetime.now(pytz.timezone("Asia/Seoul")).time()
    # 예측 가능한 구간: 오전 05:01 ~ 오후 22:29
    return time(5, 0) <= now_kst < time(22, 30)
today_kst = datetime.now(pytz.timezone("Asia/Seoul")).date()

if is_kst_predict_time():
    # 어제까지의 종가로 오늘 예측 가능
    combined_df = combined_df[combined_df["Date"] <= today_kst]
else:
    # 장이 아직 안 열렸거나 진행 중 → 전날까지만 사용
    combined_df = combined_df[combined_df["Date"] < today_kst]

MMM...
AOS...
ABT...
ABBV...
ACN...
ADBE...
AMD...
AES...
AFL...
A...
APD...
ABNB...
AKAM...
ALB...
ARE...
ALGN...
ALLE...
LNT...
ALL...
GOOGL...
GOOG...
MO...
AMZN...
AMCR...
AEE...
AEP...
AXP...
AIG...
AMT...
AWK...
AMP...
AME...
AMGN...
APH...
ADI...
ANSS...
AON...
APA...
APO...
AAPL...
AMAT...
APTV...
ACGL...
ADM...
ANET...
AJG...
AIZ...
T...
ATO...
ADSK...
ADP...
AZO...
AVB...
AVY...
AXON...
BKR...
BALL...
BAC...
BAX...
BDX...
BRK-B...
BBY...
TECH...
BIIB...
BLK...
BX...
BK...
BA...
BKNG...
BSX...
BMY...
AVGO...
BR...
BRO...
BF-B...
BLDR...
BG...
BXP...
CHRW...
CDNS...
CZR...
CPT...
CPB...
COF...
CAH...
KMX...
CCL...
CARR...
CAT...
CBOE...
CBRE...
CDW...
COR...
CNC...
CNP...
CF...
CRL...
SCHW...
CHTR...
CVX...
CMG...
CB...
CHD...
CI...
CINF...
CTAS...
CSCO...
C...
CFG...
CLX...
CME...
CMS...
KO...
CTSH...
COIN...
CL...
CMCSA...
CAG...
COP...
ED...
STZ...
CEG...
COO...
CPRT...
GLW...
CPAY...
CTVA...
CSGP...
COST...
CTRA...
CRWD...
CCI...
CSX...
CMI...
CVS...
DHR...
DRI...
DVA...
DA

In [4]:
combined_df.isnull().sum()

,0
Date,0
Open,0
High,0
Low,0
Close,0
Volume,0
Dividends,0
Stock Splits,0
Ticker,0
RSI,0


In [5]:
combined_df = combined_df.dropna()

In [6]:
import os
import pandas as pd
from datetime import date
from google.colab import drive
from datetime import datetime
import pytz

# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

# 2. 오늘 날짜
seoul_tz = pytz.timezone("Asia/Seoul")
today = datetime.now(seoul_tz).strftime("%Y-%m-%d")
file_path = f"/content/drive/Othercomputers/내 Mac/Desktop/hateslop/프로젝트/차트데이터/({today})차트.csv"

combined_df.to_csv(file_path, index=False)
print("✅ 저장 완료:", file_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 저장 완료: /content/drive/Othercomputers/내 Mac/Desktop/hateslop/프로젝트/차트데이터/(2025-05-31)차트.csv


다음날 검증할때


In [7]:
from datetime import datetime, timedelta
import pytz

# 서울 기준 어제 날짜 구하기
seoul_tz = pytz.timezone("Asia/Seoul")
yesterday = (datetime.now(seoul_tz) - timedelta(days=1)).date()
# 날짜 필터링
ccd = combined_df[combined_df["Date"] == yesterday]

# 결과 저장할 리스트
results = []

# 필터링된 ccd에서 Ticker와 Target 가져와서 변환
for i, row in ccd.iterrows():
    ticker = row["Ticker"]
    prob = row["Target"]
    kor_name = ticker_names.get(ticker, ticker)  # 한국어 이름 매핑
    results.append({
        "Ticker": ticker,
        "종목": kor_name,
        "상승여부": "상승" if prob == 1 else "하락"
    })

# 결과 DataFrame 생성
df_val = pd.DataFrame(results)
df_val = df_val.iloc[:,1:4]
df_val

# 2. 오늘 날짜
seoul_tz = pytz.timezone("Asia/Seoul")
today = datetime.now(seoul_tz).strftime("%Y-%m-%d")
file_path = f"/content/drive/Othercomputers/내 Mac/Desktop/hateslop/프로젝트/검증데이터/({today})검증.csv"

df_val.to_csv(file_path, index=False)
print("✅ 저장 완료:", file_path)

✅ 저장 완료: /content/drive/Othercomputers/내 Mac/Desktop/hateslop/프로젝트/검증데이터/(2025-05-31)검증.csv


# 데이터셋 분리

In [ ]:
data = combined_df
features = [
    'RSI', 'BB_Dist', 'MACD_6_13_4', 'MACDh_6_13_4', 'MACDs_6_13_4',
    'STOCHk_5_3_1', 'STOCHd_5_3_1', 'OBV','ATR',
    'Golden_Cross' ,'ADX', 'CCI', 'ROC', 'CMF','ADL'
]

target = 'Target'
X = data[features]
y = data[target]
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False, test_size=0.2)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((239112, 15), (59779, 15), (239112,), (59779,))

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.8041451867438191


In [ ]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 4000),
        'max_depth': trial.suggest_int('max_depth', 8, 16),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 300),
        'gamma': trial.suggest_int('gamma', 1, 3),
        'learning_rate': 0.01,
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0, step=0.1),
        'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-3, 10.0),
        'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-3, 10.0),
        'subsample': trial.suggest_categorical('subsample', [0.6, 0.7, 0.8, 1.0]),
        'random_state': 42,
        'use_label_encoder': False,
        'eval_metric': 'mlogloss'
    }
    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(accuracy)
    return accuracy

study = optuna.create_study(direction='maximize')  # accuracy이므로 maximize
study.optimize(objective, n_trials=50)

print("Best trial:")
print("  Value: ", study.best_trial.value)
from xgboost import XGBClassifier
best_params = study.best_params
final_model = XGBClassifier(random_state=42, **best_params)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy = {accuracy:.2f}')


[I 2025-05-28 08:32:20,316] A new study created in memory with name: no-name-f2683402-4cee-4900-81eb-f271592bd8c2
<ipython-input-36-4a088c71a5af>:13: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-3, 10.0),
<ipython-input-36-4a088c71a5af>:14: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-3, 10.0),
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [08:32:20] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
[I 2025-05-28 08:32:27,837] Trial 0 finished with value: 0

0.7984007226859829


<ipython-input-36-4a088c71a5af>:13: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-3, 10.0),
<ipython-input-36-4a088c71a5af>:14: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-3, 10.0),
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [08:32:28] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
[I 2025-05-28 08:32:52,316] Trial 1 finished with value: 0.8310554226541981 and parameters: {'n_estimators': 3350, 'max_depth': 14, 'min_child_weight': 268, 'gamma': 3, 'co

0.8310554226541981


<ipython-input-36-4a088c71a5af>:13: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-3, 10.0),
<ipython-input-36-4a088c71a5af>:14: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-3, 10.0),
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [08:32:52] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
[I 2025-05-28 08:33:18,991] Trial 2 finished with value: 0.8367766866855145 and parameters: {'n_estimators': 3292, 'max_depth': 8, 'min_child_weight': 18, 'gamma': 2, 'cols

...

0.8389179784867089


<ipython-input-36-4a088c71a5af>:13: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-3, 10.0),
<ipython-input-36-4a088c71a5af>:14: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-3, 10.0),
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [08:57:36] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
[I 2025-05-28 08:58:05,893] Trial 48 finished with value: 0.8385332151161818 and parameters: {'n_estimators': 3334, 'max_depth': 12, 'min_child_weight': 39, 'gamma': 3, 'co

0.8385332151161818


<ipython-input-36-4a088c71a5af>:13: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-3, 10.0),
<ipython-input-36-4a088c71a5af>:14: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-3, 10.0),
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [08:58:06] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
[I 2025-05-28 08:58:30,555] Trial 49 finished with value: 0.8351874466768155 and parameters: {'n_estimators': 3980, 'max_depth': 10, 'min_child_weight': 93, 'gamma': 2, 'co

0.8351874466768155
Best trial:
  Value:  0.8395369456479918
Accuracy = 0.83


In [ ]:
import xgboost as xgb

model = xgb.XGBClassifier(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.8213361388742767


In [ ]:
from google.colab import drive
final_model.save_model("/content/drive/Othercomputers/내 Mac/Desktop/hateslop/프로젝트/예측모델/xgb_model.json")


# 항목별 학습 그냥 해본건 무시

In [9]:
import pandas as pd
import yfinance as yf
import pandas_ta as ta
from datetime import datetime
import pytz

#오늘 날짜
today = datetime.now().date()
result_list = []

for ticker1 in ticker_Group:
    print(f"{ticker1}...")
    ticker = yf.Ticker(ticker1)
    df = ticker.history(period="2mo")
    # 기술적 지표 (단기형)
    df["RSI"] = ta.rsi(df["Close"], length=5).shift(1)
    bb = ta.bbands(df["Close"], length=10, std=2.0)
    for col in bb.columns:
        df[col] = bb[col].shift(1)
    df['SMA5'] = df['Close'].rolling(window=5).mean().shift(1)
    df['SMA10'] = df['Close'].rolling(window=10).mean().shift(1)
    df['BB_Dist'] = df['Close'] - df['BBL_10_2.0']

    macd = ta.macd(df["Close"], fast=6, slow=13, signal=4)
    for col in macd.columns:
        df[col] = macd[col].shift(1)

    stoch = ta.stoch(df["High"], df["Low"], df["Close"], k=5, d=3, smooth_k=1)
    for col in stoch.columns:
        df[col] = stoch[col].shift(1)

    df["OBV"] = ta.obv(df["Close"], df["Volume"]).shift(1)
    df["ATR"] = ta.atr(df["High"], df["Low"], df["Close"], length=14).shift(1)
    df["Golden_Cross"] = (
        (df["SMA5"].shift(1) < df["SMA10"].shift(1)) &
        (df["SMA5"] >= df["SMA10"])
    ).astype(int)
    df["ADX"] = ta.adx(df["High"], df["Low"], df["Close"]).iloc[:, 0].shift(1)
    df["CCI"] = ta.cci(df["High"], df["Low"], df["Close"], length=20).shift(1)
    df["ROC"] = ta.roc(df["Close"], length=10).shift(1)
    df["CMF"] = ta.cmf(df["High"], df["Low"], df["Close"], df["Volume"], length=20).shift(1)
    df["ADL"] = ta.ad(df["High"], df["Low"], df["Close"], df["Volume"]).shift(1)

    # 최신 행만
    last_row = df.iloc[-1]
    result_list.append({
        'Ticker': ticker1,
        'Date': today,
        'RSI': last_row['RSI'],
        'BB_Dist': last_row['BB_Dist'],
        'MACD_6_13_4': last_row['MACD_6_13_4'],
        'MACDh_6_13_4': last_row['MACDh_6_13_4'],
        'MACDs_6_13_4': last_row['MACDs_6_13_4'],
        'STOCHk_5_3_1': last_row['STOCHk_5_3_1'],
        'STOCHd_5_3_1': last_row['STOCHd_5_3_1'],
        'OBV': last_row['OBV'],
        'ATR': last_row['ATR'],
        'Golden_Cross': last_row['Golden_Cross'],
        'ADX': last_row['ADX'],
        'CCI': last_row['CCI'],
        'ROC': last_row['ROC'],
        'CMF': last_row['CMF'],
        'ADL': last_row['ADL']
    })

# 데이터프레임으로 변환
final_df = pd.DataFrame(result_list)

MMM...
AOS...
ABT...
ABBV...
ACN...
ADBE...
AMD...
AES...
AFL...
A...
APD...
ABNB...
AKAM...
ALB...
ARE...
ALGN...
ALLE...
LNT...
ALL...
GOOGL...
GOOG...
MO...
AMZN...
AMCR...
AEE...
AEP...
AXP...
AIG...
AMT...
AWK...
AMP...
AME...
AMGN...
APH...
ADI...
ANSS...
AON...
APA...
APO...
AAPL...
AMAT...
APTV...
ACGL...
ADM...
ANET...
AJG...
AIZ...
T...
ATO...
ADSK...
ADP...
AZO...
AVB...
AVY...
AXON...
BKR...
BALL...
BAC...
BAX...
BDX...
BRK-B...
BBY...
TECH...
BIIB...
BLK...
BX...
BK...
BA...
BKNG...
BSX...
BMY...
AVGO...
BR...
BRO...
BF-B...
BLDR...
BG...
BXP...
CHRW...
CDNS...
CZR...
CPT...
CPB...
COF...
CAH...
KMX...
CCL...
CARR...
CAT...
CBOE...
CBRE...
CDW...
COR...
CNC...
CNP...
CF...
CRL...
SCHW...
CHTR...
CVX...
CMG...
CB...
CHD...
CI...
CINF...
CTAS...
CSCO...
C...
CFG...
CLX...
CME...
CMS...
KO...
CTSH...
COIN...
CL...
CMCSA...
CAG...
COP...
ED...
STZ...
CEG...
COO...
CPRT...
GLW...
CPAY...
CTVA...
CSGP...
COST...
CTRA...
CRWD...
CCI...
CSX...
CMI...
CVS...
DHR...
DRI...
DVA...
DA

In [10]:
final_df

,Ticker,Date,RSI,BB_Dist,MACD_6_13_4,MACDh_6_13_4,MACDs_6_13_4,STOCHk_5_3_1,STOCHd_5_3_1,OBV,ATR,Golden_Cross,ADX,CCI,ROC,CMF,ADL
0,MMM,2025-05-31,55.780936,2.472205,1.150315,-0.368391,1.518706,80.270369,51.218248,22084100.0,3.536173,0.0,13.690967,50.025254,1.678952,0.197546,6.605289e+06
1,AOS,2025-05-31,30.652163,0.621529,-0.850428,-0.413019,-0.437410,22.134452,28.585425,1080400.0,1.754965,0.0,18.370174,-205.819523,-5.296262,-0.003435,2.471250e+06
2,ABT,2025-05-31,52.716509,3.165029,0.066351,-0.067468,0.133819,70.968059,47.465645,26119900.0,2.577698,0.0,10.432026,-10.278419,3.336970,0.022039,7.407707e+06
3,ABBV,2025-05-31,56.659111,5.292125,-0.287927,0.304805,-0.592732,82.615309,75.430887,6831300.0,5.105260,0.0,25.025867,-23.618985,4.610005,0.114871,4.731397e+07
4,ACN,2025-05-31,57.464454,6.224651,1.240007,-0.342550,1.582557,87.102557,63.901882,15821200.0,6.405648,0.0,20.719722,37.924095,-0.836426,0.153116,1.907535e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498,XYL,2025-05-31,54.883659,2.498234,0.626078,-0.195937,0.822015,69.784561,62.153465,3644400.0,2.315901,0.0,29.634605,33.092285,0.487296,-0.133407,2.421793e+06
499,YUM,2025-05-31,37.797322,1.876150,-0.842057,-0.157424,-0.684633,48.463970,25.324285,-5773200.0,2.621695,0.0,13.629950,-118.207327,-0.284817,0.013039,9.838501e+06
500,ZBRA,2025-05-31,55.930921,6.687590,3.859900,-1.066247,4.926147,75.621569,74.961760,3212200.0,9.290817,0.0,28.270288,42.178619,-2.262168,0.198631,3.438167e+06
501,ZBH,2025-05-31,45.795558,1.639411,-1.049413,0.096377,-1.145790,70.720668,59.754338,13417800.0,2.659695,0.0,33.701003,-76.113355,-2.087491,-0.039740,2.774612e+06


In [11]:
features = [
    'RSI', 'BB_Dist', 'MACD_6_13_4', 'MACDh_6_13_4', 'MACDs_6_13_4',
    'STOCHk_5_3_1', 'STOCHd_5_3_1', 'OBV','ATR',
    'Golden_Cross','ADX', 'CCI', 'ROC', 'CMF','ADL'
]

test = final_df[features]
test

,RSI,BB_Dist,MACD_6_13_4,MACDh_6_13_4,MACDs_6_13_4,STOCHk_5_3_1,STOCHd_5_3_1,OBV,ATR,Golden_Cross,ADX,CCI,ROC,CMF,ADL
0,55.780936,2.472205,1.150315,-0.368391,1.518706,80.270369,51.218248,22084100.0,3.536173,0.0,13.690967,50.025254,1.678952,0.197546,6.605289e+06
1,30.652163,0.621529,-0.850428,-0.413019,-0.437410,22.134452,28.585425,1080400.0,1.754965,0.0,18.370174,-205.819523,-5.296262,-0.003435,2.471250e+06
2,52.716509,3.165029,0.066351,-0.067468,0.133819,70.968059,47.465645,26119900.0,2.577698,0.0,10.432026,-10.278419,3.336970,0.022039,7.407707e+06
3,56.659111,5.292125,-0.287927,0.304805,-0.592732,82.615309,75.430887,6831300.0,5.105260,0.0,25.025867,-23.618985,4.610005,0.114871,4.731397e+07
4,57.464454,6.224651,1.240007,-0.342550,1.582557,87.102557,63.901882,15821200.0,6.405648,0.0,20.719722,37.924095,-0.836426,0.153116,1.907535e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498,54.883659,2.498234,0.626078,-0.195937,0.822015,69.784561,62.153465,3644400.0,2.315901,0.0,29.634605,33.092285,0.487296,-0.133407,2.421793e+06
499,37.797322,1.876150,-0.842057,-0.157424,-0.684633,48.463970,25.324285,-5773200.0,2.621695,0.0,13.629950,-118.207327,-0.284817,0.013039,9.838501e+06
500,55.930921,6.687590,3.859900,-1.066247,4.926147,75.621569,74.961760,3212200.0,9.290817,0.0,28.270288,42.178619,-2.262168,0.198631,3.438167e+06
501,45.795558,1.639411,-1.049413,0.096377,-1.145790,70.720668,59.754338,13417800.0,2.659695,0.0,33.701003,-76.113355,-2.087491,-0.039740,2.774612e+06


In [ ]:
y_pred = final_model.predict(test)
y_proba = final_model.predict_proba(test)

In [ ]:
import pandas as pd

results = []

for i, ticker in enumerate(ticker_Group):
    kor_name = ticker_names.get(ticker, ticker)
    pred = y_pred[i]
    prob = y_proba[i][1] * 100
    results.append({
        "Ticker": ticker,
        "종목": kor_name,
        "상승확률": round(prob, 2),
        "상승여부": "상승" if prob > 50 else "하락"
    })

df_result = pd.DataFrame(results)
df_result = df_result.iloc[:,1:4]
df_result

,종목,상승확률,상승여부
0,3M,48.14,하락
1,A- O- Smith,99.98,상승
2,Abbott Laboratories,99.85,상승
3,AbbVie,99.24,상승
4,Accenture,94.85,상승
...,...,...,...
498,Xylem Inc-,70.47,상승
499,Yum! Brands,43.53,하락
500,Zebra Technologies,80.92,상승
501,Zimmer Biomet,93.80,상승


In [ ]:
import os
import pandas as pd
from datetime import date
from google.colab import drive
# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

# 2. 오늘 날짜
today = date.today().strftime("%Y-%m-%d")
file_path = f"/content/drive/Othercomputers/내 Mac/Desktop/hateslop/프로젝트/예측데이터/({today})예측.csv"

df_result.to_csv(file_path, index=False)
print("✅ 저장 완료:", file_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 저장 완료: /content/drive/Othercomputers/내 Mac/Desktop/hateslop/프로젝트/예측데이터/(2025-05-27)예측.csv


In [ ]:
import os
import pandas as pd
from datetime import date
from google.colab import drive
# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

# 2. 오늘 날짜
seoul_tz = pytz.timezone("Asia/Seoul")
today = datetime.now(seoul_tz).strftime("%Y-%m-%d")
file_path = f"/content/drive/Othercomputers/내 Mac/Desktop/hateslop/프로젝트/테스트데이터/({today})테스트.csv"

test.to_csv(file_path, index=False)
print("✅ 저장 완료:", file_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 저장 완료: /content/drive/Othercomputers/내 Mac/Desktop/hateslop/프로젝트/테스트데이터/(2025-05-30)테스트.csv
